# Start Eval

Notes for what to do:


*   Import files (test folds and result json files)
*   Parse through files to get individual groups and their recommendations and a list of tuples
*   Run eval function on results using test matrix
*   Comput for 5 fold and take mean of metrics for that



## Import files and packages

In [15]:
import pandas as pd
import numpy as np
import json
from google.colab import drive
import math
import os
from collections import defaultdict


In [16]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Get recommendation lists with their groups

In [17]:
file_path = '/content/drive/MyDrive/Replication/Replication Outputs/diverse_groups-100_size-8_fold_1_list-size-3_a_j_0.5_greedy_recommendations_20250503_003105.json'

with open(file_path, 'r') as json_file:
    data = json.load(json_file)

for key, value_list in data.items():
    ids = key.split(',')  # split the key into individual string IDs
    ids = [x for x in ids]  # convert to integers
    print(f"Group IDs: {ids} => Recommended Items: {value_list}")

# print(data)

Group IDs: ['1305', '1309', '1318', '1346', '1357', '2911', '4273', '6034'] => Recommended Items: [356, 1219, 32]
Group IDs: ['324', '665', '1548', '1582', '4128', '4151', '4665', '5006'] => Recommended Items: [913, 2706, 50]
Group IDs: ['46', '47', '1305', '3338', '4584', '4602', '4923', '5332'] => Recommended Items: [527, 1193, 110]
Group IDs: ['1488', '1536', '1933', '5834', '5849', '5859', '5891', '5919'] => Recommended Items: [858, 260, 321]
Group IDs: ['2357', '2363', '2373', '2375', '2388', '4110', '4257', '6034'] => Recommended Items: [593, 858, 1248]
Group IDs: ['1123', '1154', '1573', '2037', '2343', '2623', '4955', '5911'] => Recommended Items: [356, 593, 1947]
Group IDs: ['1043', '2356', '2357', '2687', '3577', '5130', '5934', '5999'] => Recommended Items: [527, 1213, 7]
Group IDs: ['16', '704', '1035', '1452', '1477', '2234', '2632', '5967'] => Recommended Items: [913, 1193, 2975]
Group IDs: ['766', '805', '835', '1231', '1810', '2123', '2834', '4413'] => Recommended Items

### Get testing matrix

In [ ]:
test_matrix = pd.read_csv('test_matrix_fold_1.csv', index_col='userId')
test_matrix.head()
test_matrix.loc[5]
int(test_matrix.loc[1305].notna().sum())


3

# Eval function

In [18]:
def evaluate(group, recommendations, test_matrix, k):
  precision_scores = []
  recall_scores = []
  f_scores = []
  ndcg_scores = []


  for user in group:
    print(f'User: {user}')

    if user not in test_matrix.index:
      print(f"User {user} not found in test_matrix.")
      continue
    #need to get true items from user's in the test set
    true_items = set(test_matrix.loc[user].dropna().index)
    print(f'True items: {true_items}')
    #if user has no test data skip user
    if not true_items:
      print(f"User {user} has no test items.")
      continue

    recommended_items = recommendations[:k]
    print(f'recommended_items { recommended_items}')
    hits = [1 if item in true_items else 0 for item in recommended_items]
    print(f'hits: {hits}')

    precision = sum(hits)/k
    recall = sum(hits)/ min(k, len(true_items))
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)


    dcg = sum([hit / math.log2(i + 2) for i, hit in enumerate(hits)])
    idcg = sum([1 / math.log2(i + 2) for i in range(min(len(true_items), k))])
    ndcg = dcg / idcg if idcg > 0 else 0

    precision_scores.append(precision)
    recall_scores.append(recall)
    f_scores.append(f1)
    ndcg_scores.append(ndcg)

  return {
      'precision': float(np.mean(precision_scores)),
      'recall': float(np.mean(recall_scores)),
      'f1': float(np.mean(f_scores)),
      'ndcg': float(np.mean(ndcg_scores))
  }

# Start testing

### Single Fold Eval

In [19]:
def evaluate_indiv_fold(group_data, test_matrix, k):
  fold_precesion_scores = []
  fold_recall_scores = []
  fold_f_scores = []
  fold_ndcg_scores = []

  for key, items in group_data.items():
    group = [int(x) for x in key.split(',')]
    recs = [str(x) for x in items]
    metrics = evaluate(group, recs, test_matrix, k)

    fold_precesion_scores.append(metrics['precision'])
    fold_recall_scores.append(metrics['recall'])
    fold_f_scores.append(metrics['f1'])
    fold_ndcg_scores.append(metrics['ndcg'])

  return {
      'precision': float(np.mean(fold_precesion_scores)),
      'recall': float(np.mean(fold_recall_scores)),
      'f1': float(np.mean(fold_f_scores)),
      'ndcg': float(np.mean(fold_ndcg_scores))
  }


# Build function for a complete eval

## separate files and build groups with them

Notes for indices of file names


*   0: group type
*   4: fold #
*   5: list size (which needs to be split)
*   7: type
*   8: lamda




In [24]:
output_folder = '/content/drive/MyDrive/Replication/Replication Outputs'
all_files = os.listdir(output_folder)

grouped_files = defaultdict(lambda: {})

for filename in all_files:
  # print(filename)
  parts = filename.split('_')
  print(parts)
  if len(parts) < 13:
      continue

  group_type = parts[0]  # 'diverse'
  list_size = parts[5]   # 'list-size-3'
  utility = parts[6]     # 'a'
  fairness = parts[7]    # 'var'
  fold_number = parts[4] # '1'
  lambda_val = parts[8]

  key = (group_type, list_size, utility, fairness, lambda_val)
  grouped_files[key][fold_number] = filename


final_groups = []

for key, fold_dict in grouped_files.items():
  if all(str(fold) in fold_dict for fold in range(1, 6)):
        group_of_files = [fold_dict[str(fold)] for fold in range(1, 6)]
        final_groups.append(group_of_files)

print(f"Found {len(final_groups)} complete 5-fold groups.")

for fname in final_groups:
    print(fname)


['No Cross Validation']


IndexError: list index out of range

In [26]:
import re
from collections import defaultdict
import os

output_folder = '/content/drive/MyDrive/Replication/Replication Outputs'
all_files = os.listdir(output_folder)

# Example regex pattern
pattern = re.compile(
    r'^(?P<group_type>\w+)_groups-100_size-8_fold_(?P<fold>\d+)_list-size-(?P<list_size>\d+)_'
    r'(?P<utility>[a-z])_(?P<fairness>[a-z]+)_(?P<lambda>\d+(?:\.\d+)?)_greedy_recommendations'
)


grouped_files = defaultdict(dict)

for filename in all_files:
    match = pattern.match(filename)
    if not match:
        print(f"Skipping: {filename}")  # Debug if needed
        continue

    gd = match.groupdict()
    key = (gd['group_type'], gd['list_size'], gd['utility'], gd['fairness'], gd['lambda'])
    grouped_files[key][gd['fold']] = filename

# Collect final groups with all 5 folds
final_groups = []

for key, fold_dict in grouped_files.items():
    if all(str(fold) in fold_dict for fold in range(1, 6)):
        group_of_files = [fold_dict[str(fold)] for fold in range(1, 6)]
        final_groups.append(group_of_files)

print(f"Found {len(final_groups)} complete 5-fold groups.")


Skipping: No Cross Validation
Found 130 complete 5-fold groups.


In [7]:
# get test files
test_folder = '/content/drive/MyDrive/Replication/Test Files'
test_filenames = sorted(os.listdir(test_folder))
# print(test_filenames)

test_matrices = []

for filename in test_filenames:
    file_path = os.path.join(test_folder, filename)
    test_matrix = pd.read_csv(file_path, index_col='userId')
    test_matrices.append(test_matrix)


5.0


## Main Driver (Binary)

In [27]:
results_by_config = {}

for group in final_groups:
  fold_metrics = []

  # need to find k group was evaluated using
  file = group[0]
  parts = file.split('_')
  val = parts[5]
  k = int(val.split('-')[2])
  # print(k)x
  for fold_idx, filename in enumerate(group):
    # need to find k val

    file_path = os.path.join(output_folder, filename)
    with open(file_path, 'r') as json_file:
        group_data = json.load(json_file)

    test_matrix = test_matrices[fold_idx]
    metrics = evaluate_indiv_fold(group_data, test_matrix, k)
    fold_metrics.append(metrics)


    avg_metrics = {
    'precision': float(np.mean([m['precision'] for m in fold_metrics])),
    'recall': float(np.mean([m['recall'] for m in fold_metrics])),
    'f1': float(np.mean([m['f1'] for m in fold_metrics])),
    'ndcg': float(np.mean([m['ndcg'] for m in fold_metrics]))
    }

    parts = group[0].split('_')
    group_type = parts[0]
    list_size = parts[5]            # e.g., 'list-size-3'
    utility = parts[6]              # e.g., 'a'
    fairness = parts[7]            # e.g., 'var'
    lambda_val = parts[8]          # e.g., '0.5'

    config_key = (group_type, list_size, utility, fairness, lambda_val)
    results_by_config[config_key] = avg_metrics


Streaming output truncated to the last 5000 lines.
User: 4169
True items: {'3061', '2951', '2269', '3914', '3641', '1044', '650', '3055', '3006', '1198', '1398', '3261', '2479', '1245', '2070', '1413', '1684', '3678', '2989', '52', '3339', '3197', '3870', '2848', '1632', '2029', '3173', '948', '1408', '1214', '2102', '3029', '2699', '2969', '1969', '2952', '3468', '1954', '2783', '913', '1732', '2916', '2575', '3160', '3258', '3619', '360', '457', '2154', '2161', '135', '2885', '1955', '2906', '3682', '1885', '3319', '3770', '149', '2944', '3728', '2527', '1381', '3072', '612', '1391', '2867', '885', '2621', '848', '460', '1256', '3152', '1917', '3593', '2117', '2535', '1355', '2467', '3668', '1672', '3725', '1590', '2246', '261', '968', '2565', '3884', '2341', '1962', '932', '3826', '852', '2903', '2476', '1208', '1243', '270', '2434', '435', '2444', '2633', '271', '2526', '3910', '275', '2792', '3267', '1121', '3292', '2126', '3255', '3896', '2006', '256', '3451', '2430', '3935', '41

In [28]:
print(results_by_config)

{('diverse', 'list-size-3', 'a', 'lm', '0.5'): {'precision': 0.08569877344877344, 'recall': 0.08628676217961932, 'f1': 0.0859339689411118, 'ndcg': 0.0881781443069475}, ('random', 'list-size-3', 'a', 'lm', '0.5'): {'precision': 0.10903571428571428, 'recall': 0.10907738095238093, 'f1': 0.10905238095238094, 'ndcg': 0.11074527658766123}, ('diverse', 'list-size-3', 'a', 'j', '0.5'): {'precision': 0.0765981584553013, 'recall': 0.07714404246547103, 'f1': 0.07681651205936921, 'ndcg': 0.08172937647572365}, ('similar', 'list-size-3', 'a', 'lm', '0.5'): {'precision': 0.12195681302824161, 'recall': 0.122251443001443, 'f1': 0.12207466501752215, 'ndcg': 0.12227325259049551}, ('random', 'list-size-3', 'a', 'j', '0.5'): {'precision': 0.10839285714285714, 'recall': 0.10843452380952381, 'f1': 0.1084095238095238, 'ndcg': 0.11047960024901445}, ('diverse', 'list-size-3', 'a', 'lm', '0.7'): {'precision': 0.08586459836459834, 'recall': 0.08645258709544423, 'f1': 0.0860997938569367, 'ndcg': 0.0882948714648092

## Tables for results



In [29]:

formatted_tables = {}

for (grouping, list_size, utility, method, lam), metrics in results_by_config.items():
    key = (grouping, list_size, utility, method)
    if key not in formatted_tables:
        # Initialize a DataFrame with metrics as index
        formatted_tables[key] = pd.DataFrame(index=['precision', 'recall', 'f1', 'ndcg'])
    for metric_name, value in metrics.items():
        formatted_tables[key].loc[metric_name, lam] = value

# Convert lambda columns to float and sort
for key, df in formatted_tables.items():
    df.columns = df.columns.astype(float)
    formatted_tables[key] = df[sorted(df.columns)]



In [30]:
from IPython.display import display

for key, df in formatted_tables.items():
    print(f"\nResults for {key}:")
    display(df)


Results for ('diverse', 'list-size-3', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.085699,0.085865,0.085615
recall,0.086287,0.086453,0.086203
f1,0.085934,0.086100,0.085851
ndcg,0.088178,0.088295,0.088119



Results for ('random', 'list-size-3', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.109036,0.109369,0.111036
recall,0.109077,0.109411,0.111077
f1,0.109052,0.109386,0.111052
ndcg,0.110745,0.110980,0.112307



Results for ('diverse', 'list-size-3', 'a', 'j'):


...

,0.5,0.7,0.9
precision,0.173989,0.173989,0.173989
recall,0.194558,0.194558,0.194558
f1,0.180532,0.180532,0.180532
ndcg,0.189938,0.189938,0.189938



Results for ('random', 'list-size-10', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.106621,0.106621,0.106621
recall,0.120252,0.120252,0.120252
f1,0.110821,0.110821,0.110821
ndcg,0.120335,0.120335,0.120335



Results for ('random', 'list-size-10', 'a', 'var'):


,0.5,0.7,0.9
precision,0.106621,0.106621,0.106621
recall,0.120252,0.120252,0.120252
f1,0.110821,0.110821,0.110821
ndcg,0.120335,0.120335,0.120335


In [31]:
output_dir = "/content/formatted_binary_tables_csv"
os.makedirs(output_dir, exist_ok=True)

for key, df in formatted_tables.items():
    grouping, list_size, utility, method = key

    filename = f"{grouping}_list-size-{list_size}_{utility}_{method}.csv"
    filepath = os.path.join(output_dir, filename)

    df.to_csv(filepath)

    print(f"Saved: {filepath}")

Saved: /content/formatted_binary_tables_csv/diverse_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_binary_tables_csv/random_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_binary_tables_csv/diverse_list-size-list-size-3_a_j.csv
Saved: /content/formatted_binary_tables_csv/similar_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_binary_tables_csv/random_list-size-list-size-3_a_j.csv
Saved: /content/formatted_binary_tables_csv/similar_list-size-list-size-3_a_j.csv
Saved: /content/formatted_binary_tables_csv/random_list-size-list-size-3_p_var.csv
Saved: /content/formatted_binary_tables_csv/random_list-size-list-size-3_p_m.csv
Saved: /content/formatted_binary_tables_csv/similar_list-size-list-size-3_p_var.csv
Saved: /content/formatted_binary_tables_csv/similar_list-size-list-size-3_p_m.csv
Saved: /content/formatted_binary_tables_csv/diverse_list-size-list-size-3_p_var.csv
Saved: /content/formatted_binary_tables_csv/diverse_list-size-list-size-3_p_lm.csv
Saved: /co

# Borda Relevance

## borda eval

In [35]:
import numpy as np
import math
import pandas as pd

def evaluate_borda(group, recommendations, test_matrix, k=10):
    precision_scores = []
    recall_scores = []
    f_scores = []
    ndcg_scores = []

    for user in group:
        if user not in test_matrix.index:
            continue

        user_row = test_matrix.loc[user].dropna()
        if user_row.empty:
            continue

        items = user_row.index.to_numpy()
        ratings = user_row.to_numpy()

        sort_indices = np.argsort(-ratings)
        sorted_items = items[sort_indices]
        sorted_scores = np.arange(len(items), 0, -1)  # Borda scores: n, n-1, ..., 1

        borda_scores = dict(zip(sorted_items, sorted_scores))

        recommended_items = recommendations[:k]
        hits = np.array([borda_scores.get(item, 0) for item in recommended_items])

        max_possible = sorted_scores[:min(k, len(sorted_scores))].sum()

        max_borda = sorted_scores[0] if sorted_scores.size > 0 else 1
        precision = hits.sum() / (k * max_borda)
        recall = hits.sum() / max_possible if max_possible > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        discounts = np.log2(np.arange(2, 2 + len(hits)))
        dcg = (hits / discounts).sum()
        ideal_hits = sorted_scores[:len(hits)]
        idcg = (ideal_hits / discounts[:len(ideal_hits)]).sum()
        ndcg = dcg / idcg if idcg > 0 else 0

        precision_scores.append(precision)
        recall_scores.append(recall)
        f_scores.append(f1)
        ndcg_scores.append(ndcg)

    return {
        'precision': float(np.mean(precision_scores)),
        'recall': float(np.mean(recall_scores)),
        'f1': float(np.mean(f_scores)),
        'ndcg': float(np.mean(ndcg_scores))
    }



## Eval single fold (borda)

In [38]:
def borda_evaluate_indiv_fold(group_data, test_matrix, k):
  fold_precesion_scores = []
  fold_recall_scores = []
  fold_f_scores = []
  fold_ndcg_scores = []

  for key, items in group_data.items():
    group = [int(x) for x in key.split(',')]
    recs = [str(x) for x in items]
    metrics = evaluate_borda(group, recs, test_matrix, k)

    fold_precesion_scores.append(metrics['precision'])
    fold_recall_scores.append(metrics['recall'])
    fold_f_scores.append(metrics['f1'])
    fold_ndcg_scores.append(metrics['ndcg'])

  return {
      'precision': float(np.mean(fold_precesion_scores)),
      'recall': float(np.mean(fold_recall_scores)),
      'f1': float(np.mean(fold_f_scores)),
      'ndcg': float(np.mean(fold_ndcg_scores))
  }

## Main Driver (Borda)

In [39]:
borda_results_by_config = {}

for group in final_groups:
  fold_metrics = []

  # need to find k group was evaluated using
  file = group[0]
  parts = file.split('_')
  val = parts[5]
  k = int(val.split('-')[2])
  # print(k)x
  for fold_idx, filename in enumerate(group):
    # need to find k val

    file_path = os.path.join(output_folder, filename)
    with open(file_path, 'r') as json_file:
        group_data = json.load(json_file)

    test_matrix = test_matrices[fold_idx]
    metrics = borda_evaluate_indiv_fold(group_data, test_matrix, k)
    fold_metrics.append(metrics)


    avg_metrics = {
    'precision': float(np.mean([m['precision'] for m in fold_metrics])),
    'recall': float(np.mean([m['recall'] for m in fold_metrics])),
    'f1': float(np.mean([m['f1'] for m in fold_metrics])),
    'ndcg': float(np.mean([m['ndcg'] for m in fold_metrics]))
    }

    parts = group[0].split('_')
    group_type = parts[0]
    list_size = parts[5]            # e.g., 'list-size-3'
    utility = parts[6]              # e.g., 'a'
    fairness = parts[7]            # e.g., 'var'
    lambda_val = parts[8]          # e.g., '0.5'

    config_key = (group_type, list_size, utility, fairness, lambda_val)
    borda_results_by_config[config_key] = avg_metrics


### Tables for results

In [40]:
formatted_tables = {}

for (grouping, list_size, utility, method, lam), metrics in borda_results_by_config.items():
    key = (grouping, list_size, utility, method)
    if key not in formatted_tables:
        formatted_tables[key] = pd.DataFrame(index=['precision', 'recall', 'f1', 'ndcg'])
    for metric_name, value in metrics.items():
        formatted_tables[key].loc[metric_name, lam] = value

for key, df in formatted_tables.items():
    df.columns = df.columns.astype(float)
    formatted_tables[key] = df[sorted(df.columns)]



In [41]:
from IPython.display import display

for key, df in formatted_tables.items():
    print(f"\nResults for {key}:")
    display(df)


Results for ('diverse', 'list-size-3', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.072208,0.072311,0.072167
recall,0.083394,0.083496,0.083353
f1,0.077119,0.077222,0.077078
ndcg,0.082923,0.082995,0.082894



Results for ('random', 'list-size-3', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.089423,0.089573,0.091268
recall,0.095104,0.095243,0.096963
f1,0.092059,0.092204,0.093909
ndcg,0.096261,0.096361,0.097681



Results for ('diverse', 'list-size-3', 'a', 'j'):


...

,0.5,0.7,0.9
precision,0.131992,0.131992,0.131992
recall,0.192619,0.192619,0.192619
f1,0.150409,0.150409,0.150409
ndcg,0.175734,0.175734,0.175734



Results for ('random', 'list-size-10', 'a', 'lm'):


,0.5,0.7,0.9
precision,0.076969,0.076969,0.076969
recall,0.117779,0.117779,0.117779
f1,0.089322,0.089322,0.089322
ndcg,0.108126,0.108126,0.108126



Results for ('random', 'list-size-10', 'a', 'var'):


,0.5,0.7,0.9
precision,0.076969,0.076969,0.076969
recall,0.117779,0.117779,0.117779
f1,0.089322,0.089322,0.089322
ndcg,0.108126,0.108126,0.108126


In [42]:
output_dir = "/content/formatted_borda_tables_csv"
os.makedirs(output_dir, exist_ok=True)

for key, df in formatted_tables.items():
    grouping, list_size, utility, method = key

    filename = f"{grouping}_list-size-{list_size}_{utility}_{method}.csv"
    filepath = os.path.join(output_dir, filename)

    df.to_csv(filepath)

    print(f"Saved: {filepath}")

Saved: /content/formatted_borda_tables_csv/diverse_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_borda_tables_csv/random_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_borda_tables_csv/diverse_list-size-list-size-3_a_j.csv
Saved: /content/formatted_borda_tables_csv/similar_list-size-list-size-3_a_lm.csv
Saved: /content/formatted_borda_tables_csv/random_list-size-list-size-3_a_j.csv
Saved: /content/formatted_borda_tables_csv/similar_list-size-list-size-3_a_j.csv
Saved: /content/formatted_borda_tables_csv/random_list-size-list-size-3_p_var.csv
Saved: /content/formatted_borda_tables_csv/random_list-size-list-size-3_p_m.csv
Saved: /content/formatted_borda_tables_csv/similar_list-size-list-size-3_p_var.csv
Saved: /content/formatted_borda_tables_csv/similar_list-size-list-size-3_p_m.csv
Saved: /content/formatted_borda_tables_csv/diverse_list-size-list-size-3_p_var.csv
Saved: /content/formatted_borda_tables_csv/diverse_list-size-list-size-3_p_lm.csv
Saved: /content/format

In [45]:
import shutil
from google.colab import files


shutil.make_archive("formatted_binary_tables", "zip", "/content/formatted_binary_tables_csv")
shutil.make_archive("formatted_borda_tables", "zip", "/content/formatted_borda_tables_csv")


# files.download("formatted_binary_tables.zip")
files.download("formatted_borda_tables.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>